# ETL — Azure SQL → `data/raw/`

Este notebook extrae las **6 fuentes crudas** del proyecto desde Azure SQL y las vuelca a CSV en `data/raw/`.

## Convención de celdas SQL (obligatoria)

Toda celda SQL de este proyecto debe:

1. Tener tipo **SQL** (en PyCharm: dropdown de lenguaje de la celda → `SQL`).
2. Apuntar al DataSource **`usecases@uaxmathfis.database.windows.net`** (selector de la barra de la celda).
3. Definir el **Output dataframe** (variable Python en el namespace del notebook), p. ej. `df_pacientes`.
4. Llevar como cabecera dos comentarios SQL que documenten la elección, redundantes pero legibles aunque la celda se exporte:

```sql
-- @datasource: usecases@uaxmathfis.database.windows.net
-- @output: df_<nombre>
SELECT ...
```

La auth contra Azure SQL es **Azure AD interactiva** (gestionada por el IDE, usuario `abarrrui`). El navegador puede abrirse la primera vez para validar el token.

## Estructura del notebook

1. **Discovery** — listar tablas y row counts → `df_tables`.
2. **Extracción** — 6 celdas SQL, una por fuente (rellenar nombres reales tras paso 1).
3. **Persistencia** — volcado a `data/raw/<nombre>.csv` y log estructurado.
4. **Validaciones mínimas** — recuento de filas, unicidad de `paciente_id`.

---
## 1. Discovery — qué hay en la base de datos

**Antes de ejecutar:** convertir la siguiente celda a tipo **SQL** en PyCharm y elegir:
- DataSource: `usecases@uaxmathfis.database.windows.net`
- Output dataframe: `df_tables`

In [1]:
%%sql
-- @datasource: usecases@uaxmathfis.database.windows.net
-- @output: df_tables
SELECT
    s.name        AS schema_name,
    t.name        AS table_name,
    SUM(p.rows)   AS row_count
FROM sys.tables  t
JOIN sys.schemas s ON t.schema_id = s.schema_id
JOIN sys.partitions p
  ON t.object_id = p.object_id
 AND p.index_id IN (0, 1)
GROUP BY s.name, t.name
ORDER BY row_count DESC;

,schema_name,table_name,row_count
0,CASOMAT_MMM,CASOMAT_MM_06_PEDIDOS,8000000
1,CASOMAT_MMM,CASOMAT_MM_07_VENTAS_LINEAS,8000000
2,DATAEX,WEBCASE_TIMEONSITE,1719937
3,DATAEX,IA_Fraudulent_ECommerce_Transaction_Data,1472952
4,DATAEX,WEBCASE_LEADORNOT,1404679
5,DATAEX,WEBCASE_MAXID,1402893
6,DATAEX,WEBCASE_CHANNEL,1402893
7,DATAEX,WEBCASE_URL,1402890
8,CASOMAT_MMM,CASOMAT_MM_01_CLIENTES,600000
9,DATAEX,IA_SENTIMIENTO,219294


In [2]:
df_tables

,schema_name,table_name,row_count
0,CASOMAT_MMM,CASOMAT_MM_06_PEDIDOS,8000000
1,CASOMAT_MMM,CASOMAT_MM_07_VENTAS_LINEAS,8000000
2,DATAEX,WEBCASE_TIMEONSITE,1719937
3,DATAEX,IA_Fraudulent_ECommerce_Transaction_Data,1472952
4,DATAEX,WEBCASE_LEADORNOT,1404679
5,DATAEX,WEBCASE_MAXID,1402893
6,DATAEX,WEBCASE_CHANNEL,1402893
7,DATAEX,WEBCASE_URL,1402890
8,CASOMAT_MMM,CASOMAT_MM_01_CLIENTES,600000
9,DATAEX,IA_SENTIMIENTO,219294


Una vez identificadas las **6 fuentes**, pegar sus nombres en el diccionario `SOURCES` de la celda de persistencia (más abajo) y rellenar las 6 celdas SQL siguientes con `SELECT * FROM <schema>.<tabla>`.

> Si encuentras un número de tablas distinto de 6, **detente y consulta**. El enunciado fija exactamente 6 fuentes.

---
## 2. Extracción — 6 fuentes

Para cada celda SQL: tipo **SQL**, DataSource `usecases@uaxmathfis.database.windows.net`, Output dataframe `df_<nombre>`.

ETL crudo: **no transformar valores** ni renombrar columnas.

### 2.1 Fuente 1 — `Bioquimica`

In [3]:
%%sql
-- @datasource: usecases@uaxmathfis.database.windows.net
-- @output: df_fuente1
SELECT *
FROM usecases.CASOCANCER.CASOCANCER_01_BIOQUIMICOS;

,paciente_id,glucosa,colesterol,trigliceridos,hemoglobina,leucocitos,plaquetas,creatinina
0,P1000000,94.66,205.32,130.31,16.62,10.08,252.46,0.84
1,P1000001,103.94,235.17,149.65,13.09,7.21,171.32,0.81
2,P1000002,131.34,138.42,166.95,15.32,6.15,331.48,0.93
3,P1000003,115.03,182.81,132.24,14.58,4.39,349.31,1.09
4,P1000004,103.97,237.70,134.30,14.44,6.11,190.03,1.20
...,...,...,...,...,...,...,...,...
49996,P1049996,118.60,167.47,266.25,15.02,7.52,308.90,0.49
49997,P1049997,106.79,197.45,92.22,12.76,4.80,198.31,0.90
49998,P1049998,83.49,159.99,225.05,12.81,10.09,206.62,1.15
49999,P1049999,96.91,244.67,168.93,12.71,6.08,290.28,1.50


### 2.2 Fuente 2 — `Clinicos`

In [4]:
%%sql
-- @datasource: usecases@uaxmathfis.database.windows.net
-- @output: df_fuente2
SELECT *
FROM usecases.CASOCANCER.CASOCANCER_02_CLINICOS;

,paciente_id,diabetes,hipertension,obesidad,cancer,enfermedad_cardiaca,asma,epoc
0,P1000000,0,0,1,0,0,0,0
1,P1000001,0,0,1,0,0,0,0
2,P1000002,1,1,0,1,0,1,0
3,P1000003,1,1,0,0,0,0,0
4,P1000004,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...
49996,P1049996,1,0,0,0,0,0,0
49997,P1049997,0,0,0,0,0,0,0
49998,P1049998,0,0,0,1,0,0,0
49999,P1049999,0,1,1,1,1,0,0


### 2.3 Fuente 3 — `Geneticos`

In [5]:
%%sql
-- @datasource: usecases@uaxmathfis.database.windows.net
-- @output: df_fuente3
SELECT *
FROM usecases.CASOCANCER.CASOCANCER_03_GENETICOS;

,paciente_id,mut_BRCA1,mut_TP53,mut_EGFR,mut_KRAS,mut_PIK3CA,mut_ALK,mut_BRAF
0,P1000000,0,0,0,0,0,0,0
1,P1000001,0,0,0,0,0,0,0
2,P1000002,1,0,0,0,0,0,0
3,P1000003,0,0,0,0,0,0,0
4,P1000004,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...
49996,P1049996,0,0,0,1,0,0,0
49997,P1049997,0,0,0,0,0,0,0
49998,P1049998,0,0,0,0,0,0,0
49999,P1049999,0,0,0,0,0,0,0


### 2.4 Fuente 4 — `Economicos`

In [6]:
%%sql
-- @datasource: usecases@uaxmathfis.database.windows.net
-- @output: df_fuente4
SELECT *
FROM usecases.CASOCANCER.CASOCANCER_04_ECONOMICOS;

,paciente_id,tipo_seguro,coste_total,coste_farmaco,num_ingresos,dias_hospital
0,P1000000,Publico,6377.03,2502.50,1,17
1,P1000001,Publico,4333.11,1571.98,0,2
2,P1000002,Publico,55907.71,20656.22,5,100
3,P1000003,Publico,7595.16,1813.78,0,8
4,P1000004,Publico,8961.75,2822.74,0,18
...,...,...,...,...,...,...
49996,P1049996,Mixto,3130.78,887.14,1,4
49997,P1049997,Mixto,5612.23,1804.97,2,15
49998,P1049998,Publico,13905.67,3694.21,2,21
49999,P1049999,Privado,40936.48,11093.46,4,70


### 2.5 Fuente 5 — `Generales`

In [7]:
%%sql
-- @datasource: usecases@uaxmathfis.database.windows.net
-- @output: df_fuente5
SELECT *
FROM usecases.CASOCANCER.CASOCANCER_05_GENERALES;

,paciente_id,fumador,alcohol,actividad_fisica,vive
0,P1000000,0,1,Moderada,1
1,P1000001,0,1,Moderada,1
2,P1000002,0,1,Moderada,1
3,P1000003,1,1,Baja,1
4,P1000004,0,1,Baja,1
...,...,...,...,...,...
49996,P1049996,0,1,Moderada,1
49997,P1049997,0,1,Alta,1
49998,P1049998,0,1,Moderada,1
49999,P1049999,0,1,Baja,0


### 2.6 Fuente 6 — `Sociodemografico`

In [8]:
%%sql
-- @datasource: usecases@uaxmathfis.database.windows.net
-- @output: df_fuente6
SELECT *
FROM usecases.CASOCANCER.CASOCANCER_06_SOCIODEMOGRAFICOS;

,paciente_id,edad,nivel_educativo,nivel_ingresos,zona,estado_civil,num_hijos,distancia_hospital_km
0,P1000000,53,Secundaria,Medio,Urbana,Casado,2,25.6
1,P1000001,66,Secundaria,Alto,Urbana,Divorciado,0,15.2
2,P1000002,33,Secundaria,Bajo,Urbana,Casado,3,43.7
3,P1000003,45,Secundaria,Bajo,Urbana,Viudo,3,0.5
4,P1000004,53,Secundaria,Bajo,Urbana,Casado,0,7.3
...,...,...,...,...,...,...,...,...
49996,P1049996,54,Universitario,Alto,Rural,Casado,0,1.9
49997,P1049997,90,Primaria,Medio,Urbana,Casado,2,2.3
49998,P1049998,50,Primaria,Alto,Urbana,Casado,3,63.3
49999,P1049999,71,Secundaria,Bajo,Urbana,Viudo,1,33.9


---
## 3. Persistencia — volcado a `data/raw/` + log

Edita el dict `SOURCES` con la pareja `<nombre_var_dataframe>: <nombre_csv>` para las 6 fuentes ejecutadas arriba.

In [9]:
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = RAW_DIR / "_extraction_log.txt"

# Edita este diccionario con los nombres reales:
#   clave  -> nombre de la variable DataFrame definida por la celda SQL correspondiente
#   valor  -> nombre del CSV de salida (sin ruta)
SOURCES: dict[str, str] = {
    "df_fuente1":       "bioquimicos.csv",
    "df_fuente2":       "clinicos.csv",
    "df_fuente3":       "geneticos.csv",
    "df_fuente4":       "economicos.csv",
    "df_fuente5":       "generales.csv",
    "df_fuente6":       "sociodemografico.csv",
}

assert len(SOURCES) in (0, 6), "Se esperan exactamente 6 fuentes (o 0 si aún no se ha hecho discovery)."

def _now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

rows_summary: list[dict] = []
with LOG_PATH.open("a", encoding="utf-8") as log:
    log.write(f"[{_now()}] START extraction\n")
    for var_name, filename in SOURCES.items():
        df = globals().get(var_name)
        if df is None:
            msg = f"[{_now()}]   {filename:<28} SKIP (variable {var_name!r} no encontrada en namespace)\n"
            log.write(msg)
            print(msg.rstrip())
            continue
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"{var_name!r} no es un DataFrame de pandas (tipo: {type(df).__name__}).")
        out_path = RAW_DIR / filename
        df.to_csv(out_path, index=False, encoding="utf-8")
        unique_pid = df["paciente_id"].is_unique if "paciente_id" in df.columns else None
        rows_summary.append({
            "fichero": filename,
            "filas": len(df),
            "columnas": df.shape[1],
            "paciente_id_unico": unique_pid,
        })
        log.write(
            f"[{_now()}]   {filename:<28} rows={len(df):<8} cols={df.shape[1]:<3} "
            f"unique_paciente_id={unique_pid}\n"
        )
    n_ok = len(rows_summary)
    log.write(f"[{_now()}] END extraction ({n_ok}/{len(SOURCES)} OK)\n\n")

pd.DataFrame(rows_summary)

,fichero,filas,columnas,paciente_id_unico
0,bioquimicos.csv,50001,8,True
1,clinicos.csv,50001,8,True
2,geneticos.csv,50001,8,True
3,economicos.csv,50001,6,True
4,generales.csv,50001,5,True
5,sociodemografico.csv,50001,8,True


---
## 4. Validaciones mínimas

Comprobaciones que tienen que pasar antes de pasar el control a `data-explorer`:

- Existen los 6 CSV en `data/raw/`.
- La tabla principal de pacientes tiene **~50.001 filas** y `paciente_id` único.
- Las tablas relacionadas tienen `paciente_id` presente (puede repetirse, multi-fila por paciente).
- Ninguna columna 100% vacía sin que se espere.

In [10]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"

csv_files = sorted(p for p in RAW_DIR.glob("*.csv"))
summary = []
for path in csv_files:
    df = pd.read_csv(path)
    summary.append({
        "fichero": path.name,
        "filas": len(df),
        "columnas": df.shape[1],
        "tiene_paciente_id": "paciente_id" in df.columns,
        "paciente_id_unico": df["paciente_id"].is_unique if "paciente_id" in df.columns else None,
        "cols_100pct_nulas": [c for c in df.columns if df[c].isna().all()],
    })

pd.DataFrame(summary)

,fichero,filas,columnas,tiene_paciente_id,paciente_id_unico,cols_100pct_nulas
0,bioquimicos.csv,50001,8,True,True,[]
1,clinicos.csv,50001,8,True,True,[]
2,economicos.csv,50001,6,True,True,[]
3,generales.csv,50001,5,True,True,[]
4,geneticos.csv,50001,8,True,True,[]
5,sociodemografico.csv,50001,8,True,True,[]
